# DE4 - Brandenburg

In [103]:
import patoolib
import pandas as pd
import pyproj
from camelsp import get_metadata
from tqdm import tqdm

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

In [3]:
input_path = get_input_path("DE4")

data_path = input_path / "Export.7z"

if not (input_path / "Export").exists():
    patoolib.extract_archive(str(data_path), outdir=str(input_path), interactive=False, verbosity=-1)
    print("Data extracted")
else:
    print("Data already extracted")

Data already extracted


## Parse data

* data is in MEZ
* Lizenz für die Daten ist https://www.govdata.de/dl-de/by-2-0
* Die Daten enthalten geprüfte und auch Rohdaten. Dazu das Feld Quality Code.
    * 40 gut
    * 80 genügend
    * 200 Rohdaten

In [40]:
raw_meta_all = pd.read_csv(input_path / "Export" / "kihope_bb_meta.csv")

# make station_no str
raw_meta_all["station_no"] = raw_meta_all["station_no"].astype(str)

# if station_no starts with a 4, add leading zero (these are the IDs in camelsp, but also on the Brandenburg website)
raw_meta_all["station_no"] = raw_meta_all["station_no"].apply(lambda x: "0"+x if x.startswith("4") else x)

# make CATCHMENT_SIZE numeric
raw_meta_all["CATCHMENT_SIZE"] = raw_meta_all["CATCHMENT_SIZE"].str.replace(" km²", "").str.replace(",", ".").astype(float)

# get ids
ids = raw_meta_all["station_no"].tolist()

raw_meta_all

,station_no,station_name,river_name,station_longitude,station_latitude,station_gauge_datum,CATCHMENT_SIZE
0,5871600,Babelsberg-Drewitz,Nuthe,13.116304,52.362773,28.511,1792.00
1,5530500,Bad Liebenwerda,Schwarze Elster,13.394420,51.519043,83.883,3154.03
2,5956000,Gadow,Löcknitz,11.625249,53.078585,16.137,465.14
3,5935206,Groß Linde,Schlatbach,11.900234,53.115633,32.263,58.95
4,5860101,Grünheide 2,Löcknitz,13.825912,52.412152,32.948,173.69
5,5935302,Helle,Kümmernitz,12.044659,53.166421,40.698,85.99
6,5935203,Lockstädt,Stepenitz,12.030087,53.201875,42.300,249.05
7,5935200,Meyenburg,Stepenitz,12.240384,53.316125,76.124,35.89
8,5861600,Märkisch Buchholz 2,Dahme,13.747132,52.107112,37.259,519.28
9,5874601,"Neue Mühle, Wehr OP",Buckau,12.484048,52.348003,31.635,351.38


In [39]:
camelsp_meta = get_metadata()

camelsp_meta[camelsp_meta["provider_id"].isin(ids)]

,camels_id,provider_id,camels_path,nuts_lvl2,federal_state,gauge_name,waterbody_name,gauge_elevation,area,x,...,q_extent_years,w_extent_years,q_w_pearson,q_w_spearman,merit_hydro_available,q_more_than_10_years,merit_area_greater_5_smaller_15000,merit_completely_within_germany,merit_difference_to_reported_area_smaller_10_percent,merit_difference_to_reported_area_smaller_20_percent
1757,DE410020,0491200,./DE4/DE410020/DE410020_data.csv,DE4,Brandenburg,"Prenzlau, Wehr UP",Ucker (Priestergraben),16.422,394.13,4.577494e+06,...,46.994521,47.515068,0.901026,0.906370,True,True,True,True,True,True
1759,DE410040,0495601,./DE4/DE410040/DE410040_data.csv,DE4,Brandenburg,"Prenzlau, Neubg. Str.",Quillow,16.312,410.72,4.576385e+06,...,50.997260,46.991781,0.467955,0.349867,True,True,True,True,False,False
1764,DE410110,5530500,./DE4/DE410110/DE410110_data.csv,DE4,Brandenburg,Bad Liebenwerda,Schwarze Elster,83.883,3154.03,4.556514e+06,...,62.005479,62.000000,0.950655,0.925531,True,True,True,True,False,False
1807,DE410610,5844300,./DE4/DE410610/DE410610_data.csv,DE4,Brandenburg,Treppendorf,Berste,48.256,316.72,4.586489e+06,...,55.000000,51.994521,0.549619,0.378242,True,True,True,True,True,True
1820,DE410740,5860101,./DE4/DE410740/DE410740_data.csv,DE4,Brandenburg,Grünheide 2,Löcknitz,32.948,173.69,4.581201e+06,...,44.906849,25.893151,0.878027,0.908758,True,True,True,True,False,False
1827,DE410810,5861600,./DE4/DE410810/DE410810_data.csv,DE4,Brandenburg,Märkisch Buchholz 2,Dahme,37.259,519.28,4.577601e+06,...,46.994521,36.983562,0.858848,0.888915,True,True,True,True,True,True
1866,DE411220,5871600,./DE4/DE411220/DE411220_data.csv,DE4,Brandenburg,Babelsberg-Drewitz,Nuthe,28.510,1792.07,4.533215e+06,...,68.008219,50.994521,0.945349,0.953086,True,True,True,True,True,True
1867,DE411230,5871800,./DE4/DE411230/DE411230_data.csv,DE4,Brandenburg,Woltersdorf II,Hammerfließ,37.813,209.94,4.541139e+06,...,54.109589,54.997260,0.901689,0.946357,True,True,True,True,True,True
1872,DE411280,5873101,./DE4/DE411280/DE411280_data.csv,DE4,Brandenburg,Trebitz,Plane,47.397,226.69,4.507876e+06,...,60.002740,17.668493,0.699270,0.653806,True,True,True,True,True,True
1883,DE411400,5874601,./DE4/DE411400/DE411400_data.csv,DE4,Brandenburg,"Neue Mühle, Wehr OP",Buckau,31.635,351.38,4.490237e+06,...,60.750685,35.687671,0.313851,0.513545,True,True,True,True,True,True


Only one ID was not already in camelsp: 5819100

In [45]:
# ids in raw_meta_all but not in camelsp_meta
set(raw_meta_all["station_no"].tolist()) - set(camelsp_meta["provider_id"].tolist())

raw_meta_all[raw_meta_all["station_no"] == (set(raw_meta_all["station_no"].tolist()) - set(camelsp_meta["provider_id"].tolist())).pop()]

,station_no,station_name,river_name,station_longitude,station_latitude,station_gauge_datum,CATCHMENT_SIZE
14,5819100,"Sachsenhausen, Teerofen",Teschendorfer Graben,13.201751,52.791997,33.333,161.05


In [88]:
def parse_station_data(id: str, aggregate_hourly: bool) -> pd.DataFrame:
    # read q and w data for the station
    data_q = pd.read_csv(input_path / "Export" / f"{id}_Q15.csv")
    data_w = pd.read_csv(input_path / "Export" / f"{id}_W15.csv")

    # make date time column from Date and Time columns
    data_q["date"] = pd.to_datetime(data_q["Date"] + " " + data_q["Time"], format="%Y-%m-%d %H:%M:%S")
    data_w["date"] = pd.to_datetime(data_w["Date"] + " " + data_w["Time"], format="%Y-%m-%d %H:%M:%S")

    # merge q and w data on date
    data = pd.merge(data_q, data_w, on="date", how="outer", suffixes=("_q", "_w"))
    data = data.sort_values("date").reset_index(drop=True)

    # select relevant columns and rename
    data = data[["date", "Q", "W"]]
    data = data.rename(columns={"Q": "discharge_vol_obs", "W": "water_level_obs"})

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    return data

parse_station_data(ids[0], aggregate_hourly=True)

,date,discharge_vol_obs,water_level_obs
0,2010-12-31 23:00:00+00:00,NaN,220.0
1,2011-01-01 00:00:00+00:00,NaN,220.0
2,2011-01-01 01:00:00+00:00,NaN,220.0
3,2011-01-01 02:00:00+00:00,NaN,220.0
4,2011-01-01 03:00:00+00:00,NaN,220.0
...,...,...,...
122732,2024-12-31 19:00:00+00:00,5.99,170.0
122733,2024-12-31 20:00:00+00:00,5.99,170.0
122734,2024-12-31 21:00:00+00:00,5.99,170.0
122735,2024-12-31 22:00:00+00:00,5.99,170.0


In [97]:
def check_metadata_consistency(camelsp_meta, raw_meta, id: str) -> dict:
    """
    Check consistency of metadata fields across different metadata sources.
    Waterbody field is only compared between raw and camelsp.
    
    """
    inconsistencies = {}
    
    fields = {
        "provider_id": {
            "camelsp": "provider_id",
            "raw": "station_no",
        },
        "gauge_name": {
            "camelsp": "gauge_name",
            "raw": "station_name"
        },
        "waterbody": {
            "camelsp": "waterbody_name",
            "raw": "river_name"
        },
        "area": {
            "camelsp": "area",
            "raw": "CATCHMENT_SIZE"
        },
        "gauge_elevation": {
            "camelsp": "gauge_elevation",
            "raw": "station_gauge_datum"
        },
        "lat": {
            "camelsp": "lat",
            "raw": "station_latitude"
        },
        "lon": {
            "camelsp": "lon",
            "raw": "station_longitude"
        }
    }

    if id not in camelsp_meta["provider_id"].values:
        print(f"ID {id} not found in camelsp metadata.")
        return None
    else:
        camelsp_row = camelsp_meta[camelsp_meta["provider_id"] == id].iloc[0]
        raw_row = raw_meta[raw_meta["station_no"] == id].iloc[0]

        for field, sources in fields.items():
            camelsp_value = camelsp_row[sources["camelsp"]]
            raw_value = raw_row[sources["raw"]]

            if field in ["area", "gauge_elevation"]:
                # allow small difference due to rounding
                if not abs(camelsp_value - raw_value) < 0.1:
                    inconsistencies[field] = (camelsp_value, raw_value)
            elif field in ["lat", "lon"]:
                # allow small difference due to rounding
                if not abs(camelsp_value - raw_value) < 0.0001:
                    inconsistencies[field] = (camelsp_value, raw_value)
            else:
                if camelsp_value != raw_value:
                    inconsistencies[field] = (camelsp_value, raw_value)
    
    return inconsistencies

for id in ids:
    check_metadata_consistency(camelsp_meta, raw_meta_all, id)

ID 5819100 not found in camelsp metadata.


Metadata in camelsp and raw data are consistent.  
We can just use the metadata from camelsp to be sure that it is exactly the same as in CAMELS-DE daily.  
We just have to add the single new station with hourly data.

In [111]:
# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DE4", add_missing=True)

    # parse data for the station
    data = parse_station_data(id, aggregate_hourly=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["station_no"] == id]

    if not camelsp_meta.empty:
        # set metadata values from camelsp metadata
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": camelsp_meta["waterbody_name"].values[0],
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
            "area_metadata": camelsp_meta["area"].values[0],
            "part_of_camelsp": True,
        }
    elif not raw_meta.empty:
        print(f"Station {id} not in camelsp metadata, using raw metadata.")

        # parse the location
        lon, lat = raw_meta["station_longitude"].values[0], raw_meta["station_latitude"].values[0]

        # from EPSG:4326 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:4326", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(lon, lat)

        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["station_name"].values[0],
            "waterbody_name": raw_meta["river_name"].values[0],
            "federal_state": "Brandenburg",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": raw_meta["station_gauge_datum"].values[0],
            "area_metadata": raw_meta["CATCHMENT_SIZE"].values[0],
            "part_of_camelsp": False,
        }
    else:
        raise ValueError(f"Station {id} has no metadata.")
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(raw_meta)
    

  0%|          | 0/19 [00:00<?, ?it/s]

 74%|███████▎  | 14/19 [00:14<00:05,  1.09s/it]

Station 5819100 not in camelsp metadata, using raw metadata.


100%|██████████| 19/19 [00:19<00:00,  1.05s/it]
